In [ ]:
!pip install -q -U datasets tokenizers huggingface_hub pyarrow safetensors
!pip install -q git+https://github.com/csebuetnlp/normalizer

import io, json, math, os, random, re, shutil, time, urllib.request
from dataclasses import dataclass, asdict
from pathlib import Path
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, IterableDataset
from huggingface_hub import HfApi, hf_hub_download, login, snapshot_download
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

HF_TOKEN   = "hf_xyz"
HF_REPO_ID = "logicwahid/Kotha-S1"
REPO_TYPE  = "model"
MODEL_NAME = "Kotha-S1"
COMPANY    = "Kotha Lab"
CREATOR    = "Md. Wahid"

DATA_DIR = Path("data")
RAW_DIR  = DATA_DIR / "raw"
PREP_DIR = DATA_DIR / "prepared"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PREP_DIR.mkdir(parents=True, exist_ok=True)

WIKI_BN_PATTERNS = ["20231101.bn/*.parquet"]
FINEWEB_PATTERNS = ["sample/10BT/000_00000.parquet", "sample/10BT/001_00000.parquet"]
BN_WEB_URL = "https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/bn.txt"
BN_WEB_GB  = 3.0
QA_JSONL = "bangladesh_1971_2026_dataset.json"
PREPARED_FILES = ["bn_wiki.parquet", "bn_web.parquet", "en_fineweb.parquet", "qa.parquet"]

MIX = {"bn_wiki": 0.20, "bn_web": 0.45, "en_fineweb": 0.30, "qa": 0.05}

VOCAB_SIZE      = 32000
TOKENIZER_NAME  = "bilingual_tokenizer.json"
SPECIAL_TOKENS  = ["<pad>", "<unk>", "<bos>", "<eos>"]
TOKENIZER_DOCS  = 400_000
DIM         = 640
N_LAYERS    = 12
N_HEADS     = 10
MAX_SEQ_LEN = 512
NORM_EPS    = 1e-5
TIE_WEIGHTS = False

BATCH_SIZE   = 8
GRAD_ACCUM   = 4
MAX_STEPS    = 50_000
WARMUP_STEPS = 500
PEAK_LR      = 3e-4
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 1.0
SAVE_EVERY   = 1000
LOG_EVERY    = 50
SEED         = 42
CHECKPOINT_NAME = "latest_checkpoint.pt"
print("Setup Config Complete.")

In [ ]:
api = HfApi(token=HF_TOKEN)
login(token=HF_TOKEN, add_to_git_credential=False)
print(f"Logged in to HF repository: {HF_REPO_ID}")

def ensure_repo(repo_id=HF_REPO_ID, repo_type=REPO_TYPE):
    try:
        api.create_repo(repo_id=repo_id, repo_type=repo_type, exist_ok=True, private=False)
        print(f"Repository ready: {repo_type}/{repo_id}")
    except Exception as e:
        print(f"Repo check or creation failed: {e}")

def repo_files(repo_id=HF_REPO_ID, repo_type=REPO_TYPE):
    try:
        return api.list_repo_files(repo_id=repo_id, repo_type=repo_type)
    except Exception:
        return []

def upload_file(local_path, path_in_repo, message=""):
    try:
        api.upload_file(path_or_fileobj=str(local_path), path_in_repo=path_in_repo,
                        repo_id=HF_REPO_ID, repo_type=REPO_TYPE,
                        commit_message=message or f"Update {path_in_repo}")
        print(f"Uploaded: {path_in_repo}")
        return True
    except Exception as e:
        print(f"Upload failed ({path_in_repo}): {e}")
        return False

ensure_repo()

In [ ]:
def data_ready():
    return all((PREP_DIR / n).exists() for n in PREPARED_FILES)

def fetch_prepared_from_hub():
    got = 0
    for f in repo_files():
        if f.startswith("prepared/") and f.endswith(".parquet"):
            name = Path(f).name
            if (PREP_DIR / name).exists():
                continue
            print(f"Downloading from HF: {f}")
            try:
                hf_hub_download(HF_REPO_ID, f, repo_type=REPO_TYPE,
                                local_dir=str(DATA_DIR), token=HF_TOKEN)
                got += 1
            except Exception as e:
                print(f"Download failed: {e}")
    return got

def fetch_qa_from_hub():
    p = Path(QA_JSONL)
    if p.exists():
        print(f"QA dataset exists locally: {p}")
        return True
    try:
        path = hf_hub_download(HF_REPO_ID, QA_JSONL, repo_type=REPO_TYPE, token=HF_TOKEN)
        shutil.copy(path, p)
        print(f"QA dataset downloaded from HF: {p}")
        return True
    except Exception as e:
        print(f"QA dataset not found: {e}")
        return False

def snapshot(repo_id, patterns, dest, repo_type="dataset"):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {repo_id} with patterns: {patterns}")
    snapshot_download(repo_id=repo_id, repo_type=repo_type, allow_patterns=patterns,
                      local_dir=str(dest), max_workers=4, token=HF_TOKEN)

def download_range(url, dest, n_bytes, chunk=1 << 20):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size >= n_bytes:
        print(f"File already exists: {dest} ({dest.stat().st_size/1e9:.2f} GB)")
        return
    req = urllib.request.Request(url, headers={"Range": f"bytes=0-{n_bytes-1}"})
    with urllib.request.urlopen(req, timeout=180) as r, open(dest, "wb") as f:
        total = 0
        while True:
            b = r.read(chunk)
            if not b:
                break
            f.write(b); total += len(b)
            if total % (256 << 20) < chunk:
                print(f"   Downloaded ... {total/1e9:.2f} GB")
    print(f"Downloaded successfully: {dest} ({dest.stat().st_size/1e9:.2f} GB)")

def download_raw():
    print("\n=== Downloading raw data from sources ===")
    snapshot("wikimedia/wikipedia", WIKI_BN_PATTERNS, RAW_DIR / "wiki_bn")
    snapshot("HuggingFaceFW/fineweb-edu", FINEWEB_PATTERNS, RAW_DIR / "fineweb")
    download_range(BN_WEB_URL, RAW_DIR / "bn_web" / "bn.txt", int(BN_WEB_GB * 1e9))

fetch_qa_from_hub()
if data_ready():
    print("Prepared data is available locally.")
else:
    n = fetch_prepared_from_hub()
    if data_ready():
        print(f"Prepared data loaded from HF repository ({n} files)")
    else:
        download_raw()

In [ ]:
ZWSP  = chr(0x200B)
BOM   = chr(0xFEFF)
BN_CH = "[" + chr(0x0980) + "-" + chr(0x09FF) + "]"

def clean_texts(texts):
    arr = pa.array(texts, type=pa.string())
    arr = pc.utf8_normalize(arr, form="NFC")
    arr = pc.replace_substring_regex(arr, ZWSP + "|" + BOM, "")
    arr = pc.replace_substring_regex(arr, "[ \	]+", " ")
    arr = pc.replace_substring_regex(arr, "\n{3,}", "\n\n")
    return pc.utf8_trim_whitespace(arr)

def _tables_parquet(files, min_chars=100):
    for fp in files:
        pf = pq.ParquetFile(str(fp))
        for batch in pf.iter_batches(batch_size=8192, columns=["text"]):
            texts = [t for t in batch.column(0).to_pylist() if t]
            if not texts:
                continue
            arr = clean_texts(texts)
            arr = pc.filter(arr, pc.greater_equal(pc.utf8_length(arr), min_chars))
            if len(arr):
                yield pa.table({"text": arr})

def _bn_char_count(arr):
    rest = pc.replace_substring_regex(arr, BN_CH, "")
    return pc.subtract(pc.utf8_length(arr), pc.utf8_length(rest))

def _tables_bn(texts, min_chars=100, min_bn_pct=50):
    arr = clean_texts(texts)
    total = pc.utf8_length(arr)
    bn = _bn_char_count(arr)
    keep = pc.and_(pc.greater_equal(total, min_chars),
                   pc.greater_equal(pc.multiply(bn, 100), pc.multiply(total, min_bn_pct)))
    arr = pc.filter(arr, keep)
    if len(arr):
        yield pa.table({"text": arr})

def _tables_txt(path, min_chars=100, min_bn_pct=50):
    buf = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            buf.append(line)
            if len(buf) >= 20_000:
                yield from _tables_bn(buf, min_chars, min_bn_pct)
                buf = []
    if buf:
        yield from _tables_bn(buf, min_chars, min_bn_pct)

_TAG_RE = re.compile(r"<(?:thought|call|response)>.*?</(?:thought|call|response)>", re.S)

def _chat_template(user, assistant):
    user = user.strip().replace(chr(10), " ")
    asst = _TAG_RE.sub("", assistant).strip()
    asst = re.sub(r"\n{3,}", "\n\n", asst)
    if user.startswith("<thought>"):
        user = user.split("</thought>", 1)[-1].strip()
    return f"[INST] {user} [/INST] {asst}"

def _tables_qa(json_path):
    out = []
    items = []
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                items = data
            elif isinstance(data, dict):
                items = data.get("data", [data])
    except Exception:
        with open(json_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    items.append(json.loads(line))
                except Exception:
                    continue
    for d in items:
        if not isinstance(d, dict):
            continue
        msgs = d.get("messages", [])
        if len(msgs) < 2:
            continue
        user = next((m.get("content", "") for m in msgs if m.get("role") == "user"), "")
        asst = next((m.get("content", "") for m in msgs if m.get("role") == "assistant"), "")
        if not user or not asst:
            continue
        tag = f"({d.get('category','bangladesh')}) " if d.get("category") else ""
        out.append(tag + _chat_template(user, asst))
    arr = pa.array(out, type=pa.string())
    print(f"QA dataset processed: {len(out):,} samples")
    yield pa.table({"text": arr})

def write_parquet(tables_iter, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    writer = None
    rows = 0
    for tbl in tables_iter:
        if writer is None:
            writer = pq.ParquetWriter(str(out_path), tbl.schema, compression="zstd")
        writer.write_table(tbl)
        rows += tbl.num_rows
    if writer is None:
        schema = pa.schema([("text", pa.string())])
        writer = pq.ParquetWriter(str(out_path), schema, compression="zstd")
    writer.close()
    size = out_path.stat().st_size / 1e6
    print(f"Created {out_path.name}: {rows:,} documents ({size:,.0f} MB)")
    return rows

def prepare_all():
    wiki  = sorted((RAW_DIR / "wiki_bn").rglob("*.parquet"))
    fine  = sorted((RAW_DIR / "fineweb").rglob("*.parquet"))
    bnweb = RAW_DIR / "bn_web" / "bn.txt"

    if not (PREP_DIR / "bn_wiki.parquet").exists() and wiki:
        print("Processing bn_wiki...")
        write_parquet(_tables_parquet(wiki), PREP_DIR / "bn_wiki.parquet")
    else:
        print("bn_wiki.parquet is ready.")

    if not (PREP_DIR / "bn_web.parquet").exists() and bnweb.exists():
        print("Processing bn_web (IndicCorpV2)...")
        write_parquet(_tables_txt(bnweb), PREP_DIR / "bn_web.parquet")
    else:
        print("bn_web.parquet is ready.")

    if not (PREP_DIR / "en_fineweb.parquet").exists() and fine:
        print("Processing en_fineweb...")
        write_parquet(_tables_parquet(fine), PREP_DIR / "en_fineweb.parquet")
    else:
        print("en_fineweb.parquet is ready.")

    if not (PREP_DIR / "qa.parquet").exists() and Path(QA_JSONL).exists():
        print("Processing qa dataset...")
        write_parquet(_tables_qa(Path(QA_JSONL)), PREP_DIR / "qa.parquet")
    elif (PREP_DIR / "qa.parquet").exists():
        print("qa.parquet is ready.")

if data_ready():
    print("Prepared parquet files exist.")
else:
    prepare_all()

In [ ]:
DATA_CARD = """# Kotha-S1 prepared dataset info.

| File | Language | Source | Role |
|---|---|---|---|
| `bn_wiki.parquet` | Bengali | `wikimedia/wikipedia` | Encyclopedic |
| `bn_web.parquet` | Bengali | `ai4bharat/IndicCorpV2` | Web / News / Books |
| `en_fineweb.parquet` | English | `HuggingFaceFW/fineweb-edu` | Educational |
| `qa.parquet` | Bilingual | Local JSONL | Bangladesh History & QA |
"""

def upload_prepared():
    existing = repo_files()
    for name in PREPARED_FILES:
        src = PREP_DIR / name
        if not src.exists():
            print(f"Missing file: {src}")
            continue
        if f"prepared/{name}" in existing:
            print(f"Already uploaded: prepared/{name}")
            continue
        upload_file(src, f"prepared/{name}", f"Add prepared dataset {name}")
    (PREP_DIR / "README.md").write_text(DATA_CARD, encoding="utf-8")
    upload_file(PREP_DIR / "README.md", "prepared/README.md", "Add dataset card")
    if Path(QA_JSONL).exists():
        upload_file(Path(QA_JSONL), "data/" + QA_JSONL, "Add raw QA dataset")

upload_prepared()

In [ ]:
SOURCES = {
    "bn_wiki":    PREP_DIR / "bn_wiki.parquet",
    "bn_web":     PREP_DIR / "bn_web.parquet",
    "en_fineweb": PREP_DIR / "en_fineweb.parquet",
}
if (PREP_DIR / "qa.parquet").exists():
    SOURCES["qa"] = PREP_DIR / "qa.parquet"
else:
    MIX.pop("qa", None)

def tokenizer_corpus(limit):
    pfs = {k: pq.ParquetFile(str(v)) for k, v in SOURCES.items()}
    its = {k: iter(pf.iter_batches(batch_size=1024, columns=["text"]))
           for k, pf in pfs.items()}
    order = list(SOURCES.keys())
    n = 0
    while n < limit:
        moved = False
        for k in order:
            try:
                batch = next(its[k])
            except StopIteration:
                continue
            moved = True
            for t in batch.column(0).to_pylist():
                if not t:
                    continue
                yield t
                n += 1
                if n >= limit:
                    return
        if not moved:
            return

def train_tokenizer():
    tok = Tokenizer(models.BPE(unk_token="<unk>", byte_fallback=True))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=SPECIAL_TOKENS,
        min_frequency=2,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )
    print("Starting tokenizer training (Byte-level BPE)...")
    tok.train_from_iterator(tokenizer_corpus(TOKENIZER_DOCS), trainer=trainer)
    size = tok.get_vocab_size()
    if size < VOCAB_SIZE * 0.9:
        print(f"Vocab size {size} is too small. Retrying with min_frequency=1...")
        trainer = trainers.BpeTrainer(
            vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS, min_frequency=1,
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=True)
        tok = Tokenizer(models.BPE(unk_token="<unk>", byte_fallback=True))
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
        tok.decoder = decoders.ByteLevel()
        tok.train_from_iterator(tokenizer_corpus(TOKENIZER_DOCS), trainer=trainer)
        size = tok.get_vocab_size()
    tok.save(TOKENIZER_NAME)
    print(f"Tokenizer trained. Vocab size: {size:,}")
    return tok

def load_or_train_tokenizer():
    try:
        path = hf_hub_download(HF_REPO_ID, TOKENIZER_NAME, repo_type=REPO_TYPE, token=HF_TOKEN)
        tok = Tokenizer.from_file(path)
        print(f"Loaded tokenizer from HF. Vocab: {tok.get_vocab_size():,}")
        return tok
    except Exception:
        print("Tokenizer not found on HF, starting new training.")
    tok = train_tokenizer()
    upload_file(TOKENIZER_NAME, TOKENIZER_NAME, f"Bilingual tokenizer ({tok.get_vocab_size()} tokens)")
    return tok

tokenizer = load_or_train_tokenizer()

for s in ["বাংলাদেশের ইতিহাস অত্যন্ত সমৃদ্ধ এবং বৈচিত্র্যপূর্ণ।",
          "Artificial intelligence is transforming every industry."]:
    ids = tokenizer.encode(s).ids
    print(f"\n'{s}'\n  → {len(ids)} tokens: {tokenizer.encode(s).tokens[:14]}")

In [ ]:
@dataclass
class ModelArgs:
    dim: int = DIM
    n_layers: int = N_LAYERS
    n_heads: int = N_HEADS
    vocab_size: int = VOCAB_SIZE
    max_seq_len: int = MAX_SEQ_LEN
    norm_eps: float = NORM_EPS

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dt = x.dtype
        x = x.float()
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * self.weight.float()).to(dt)

def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end)
    freqs = torch.outer(t, freqs).float()
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor):
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2).to(x.device)
    x_rotated = torch.view_as_real(x_complex * freqs_cis).flatten(3)
    return x_rotated.type_as(x)

class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.head_dim = args.dim // args.n_heads
        self.wq = nn.Linear(args.dim, args.dim, bias=False)
        self.wk = nn.Linear(args.dim, args.dim, bias=False)
        self.wv = nn.Linear(args.dim, args.dim, bias=False)
        self.wo = nn.Linear(args.dim, args.dim, bias=False)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim)
        q = apply_rotary_emb(q, freqs_cis[:T])
        k = apply_rotary_emb(k, freqs_cis[:T])
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.wo(out)

class FeedForward(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        hidden_dim = int(2 * (4 * args.dim) / 3)
        self.w1 = nn.Linear(args.dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, args.dim, bias=False)
        self.w3 = nn.Linear(args.dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class TransformerBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.attention = Attention(args)
        self.feed_forward = FeedForward(args)
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor):
        h = x + self.attention(self.attention_norm(x), freqs_cis)
        return h + self.feed_forward(self.ffn_norm(h))

class BilingualLLaMA(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
        self.layers = nn.ModuleList([TransformerBlock(args) for _ in range(args.n_layers)])
        self.norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)
        if TIE_WEIGHTS:
            self.output.weight = self.tok_embeddings.weight
        freqs_cis = precompute_freqs_cis(args.dim // args.n_heads, args.max_seq_len * 2)
        self.register_buffer("freqs_cis", freqs_cis, persistent=False)

    def forward(self, tokens: torch.Tensor):
        h = self.tok_embeddings(tokens)
        for layer in self.layers:
            h = layer(h, self.freqs_cis)
        h = self.norm(h)
        return self.output(h)

args = ModelArgs(vocab_size=tokenizer.get_vocab_size())
model = BilingualLLaMA(args)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Model Parameters: {total_params:,}")
print(f"Vocab: {args.vocab_size:,} | Dim: {args.dim} | Layers: {args.n_layers} | Heads: {args.n_heads}")

In [ ]:
class KothaDataset(IterableDataset):
    def __init__(self, tokenizer, seq_len=MAX_SEQ_LEN, seed=SEED, mix=None, pos=None):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.seed = seed
        self.mix = dict(mix or MIX)
        self.names = list(self.mix.keys())
        self.weights = [self.mix[k] for k in self.names]
        self.pos = {k: {"epoch": 0, "offset": 0} for k in self.names}
        if pos:
            for k, v in pos.items():
                if k in self.pos:
                    self.pos[k] = {"epoch": int(v.get("epoch", 0)),
                                   "offset": int(v.get("offset", 0))}
        self.eos_id = tokenizer.token_to_id("<eos>")

    def _stream(self, key):
        path = SOURCES[key]
        pf = pq.ParquetFile(str(path))
        skip = self.pos[key]["offset"]
        start_rg = 0
        if skip > 0:
            for rg in range(pf.num_row_groups):
                n = pf.metadata.row_group(rg).num_rows
                if skip >= n:
                    skip -= n
                else:
                    start_rg = rg
                    break
            else:
                start_rg = pf.num_row_groups
        if start_rg >= pf.num_row_groups:
            return
        for rg in range(start_rg, pf.num_row_groups):
            inner = skip if rg == start_rg else 0
            for batch in pf.iter_batches(batch_size=2048, row_groups=[rg], columns=["text"]):
                texts = batch.column(0).to_pylist()
                if inner:
                    texts = texts[inner:]
                    inner = 0
                for t in texts:
                    if t:
                        yield t

    def __iter__(self):
        rng = random.Random(self.seed)
        iters = {k: self._stream(k) for k in self.names}
        buffer = []
        while True:
            block = rng.choices(self.names, weights=self.weights, k=1024)
            for k in block:
                row = next(iters[k], None)
                if row is None:
                    self.pos[k]["epoch"] += 1
                    self.pos[k]["offset"] = 0
                    iters[k] = self._stream(k)
                    row = next(iters[k], None)
                    if row is None:
                        continue
                self.pos[k]["offset"] += 1
                buffer.extend(self.tokenizer.encode(row).ids + [self.eos_id])
                while len(buffer) >= self.seq_len + 1:
                    chunk = buffer[: self.seq_len + 1]
                    buffer = buffer[self.seq_len:]
                    yield (torch.tensor(chunk[:-1], dtype=torch.long),
                           torch.tensor(chunk[1:], dtype=torch.long))

    def stats(self):
        return {k: f"ep{v['epoch']} +{v['offset']:,}" for k, v in self.pos.items()}

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU not found! Enable GPU T4 in Kaggle or Colab accelerator.")

_AMP_NEW = hasattr(torch.amp, "GradScaler")
def autocast_ctx():
    if _AMP_NEW:
        return torch.amp.autocast("cuda", dtype=torch.float16)
    return torch.cuda.amp.autocast(dtype=torch.float16)

scaler = torch.amp.GradScaler("cuda") if _AMP_NEW else torch.cuda.amp.GradScaler()

model = model.to("cuda")
if torch.cuda.device_count() > 1:
    print(f"{torch.cuda.device_count()} GPUs available. Using nn.DataParallel.")
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))

dataset = KothaDataset(tokenizer, seq_len=args.max_seq_len, seed=SEED, mix=MIX)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)

def lr_at(step):
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    p = min(1.0, max(0.0, p))
    return PEAK_LR * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * p)))

def save_checkpoint(model, optimizer, scaler, step, loss, dataset):
    raw = model.module if hasattr(model, "module") else model
    ckpt = {
        "step": step,
        "model_state_dict": raw.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": float(loss),
        "model_args": asdict(args),
        "data_pos": {k: dict(v) for k, v in dataset.pos.items()},
        "mix": dict(MIX),
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    torch.save(ckpt, CHECKPOINT_NAME)
    print(f"Local Checkpoint saved: {CHECKPOINT_NAME} (step {step})")
    upload_file(CHECKPOINT_NAME, CHECKPOINT_NAME,
                f"Kotha-S1 checkpoint | step {step} | loss {loss:.4f}")

def load_checkpoint(model, optimizer, scaler, dataset):
    path = None
    try:
        path = hf_hub_download(HF_REPO_ID, CHECKPOINT_NAME, repo_type=REPO_TYPE, token=HF_TOKEN)
        print("Loaded checkpoint from HF.")
    except Exception:
        if Path(CHECKPOINT_NAME).exists():
            path = CHECKPOINT_NAME
            print("Loaded local checkpoint.")
    if path is None:
        print("No checkpoint found. Starting training from scratch.")
        return 0
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    raw = model.module if hasattr(model, "module") else model
    raw.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    if "data_pos" in ckpt:
        dataset.pos.update({k: dict(v) for k, v in ckpt["data_pos"].items()})
    print(f"Resumed from step {ckpt['step']} | Loss: {ckpt.get('loss', float('nan')):.4f}")
    print(f"Data positions: {dataset.stats()}")
    return int(ckpt["step"])

current_step = load_checkpoint(model, optimizer, scaler, dataset)
print(f"\nTraining active: step {current_step} / {MAX_STEPS} | Batch size: {BATCH_SIZE * GRAD_ACCUM}")

model.train()
data_iter = iter(dataloader)
t0 = time.time()

while current_step < MAX_STEPS:
    lr = lr_at(current_step)
    for g in optimizer.param_groups:
        g["lr"] = lr
    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for _ in range(GRAD_ACCUM):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            x, y = next(data_iter)
        x = x.to("cuda", non_blocking=True)
        y = y.to("cuda", non_blocking=True)
        with autocast_ctx():
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, args.vocab_size), y.view(-1))
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_loss += loss.item()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    current_step += 1
    if current_step % LOG_EVERY == 0:
        el = time.time() - t0
        tps = (LOG_EVERY * BATCH_SIZE * GRAD_ACCUM * args.max_seq_len) / max(el, 1e-6)
        print(f"step {current_step:>6}/{MAX_STEPS} | loss {accum_loss:.4f} | lr {lr:.2e} | "
              f"{el:.0f}s | {tps:,.0f} tok/s | {dataset.stats()}")
        t0 = time.time()
    if current_step % SAVE_EVERY == 0:
        save_checkpoint(model, optimizer, scaler, current_step, accum_loss, dataset)

save_checkpoint(model, optimizer, scaler, current_step, accum_loss, dataset)
print("Training complete!")

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=120, temperature=0.8, top_p=0.9):
    raw = model.module if hasattr(model, "module") else model
    was_training = model.training
    model.eval()

    bos = tokenizer.token_to_id("<bos>")
    eos = tokenizer.token_to_id("<eos>")
    ids = ([bos] if bos is not None else []) + tokenizer.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device="cuda")

    for _ in range(max_new_tokens):
        x_cond = x[:, -args.max_seq_len:]
        with autocast_ctx():
            logits = raw(x_cond)
        logits = logits[:, -1, :].float() / max(temperature, 1e-5)
        probs = F.softmax(logits, dim=-1)

        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cdf = torch.cumsum(sorted_probs, dim=-1)
        sorted_probs = sorted_probs.masked_fill(cdf - sorted_probs > top_p, 0.0)
        sorted_probs = sorted_probs / sorted_probs.sum()
        nxt = sorted_idx.gather(1, torch.multinomial(sorted_probs, num_samples=1))
        x = torch.cat((x, nxt), dim=1)
        if nxt.item() == eos:
            break

    if was_training:
        model.train()
    return tokenizer.decode(x[0].tolist())

print("--- Bengali generation demo ---")
print(generate("বাংলাদেশের ইতিহাস অত্যন্ত"))
print("\n--- English generation demo ---")
print(generate("Artificial intelligence is the"))
print("\n--- Prompt engineering demo ---")
print(generate("প্রশ্ন: বাংলাদেশের রাজধানী কোন?\nউত্তর:"))

In [ ]:
def export_transformers_format(model, tokenizer, args, out_dir="hf_export"):
    from safetensors.torch import save_file
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    raw = model.module if hasattr(model, "module") else model
    sd = raw.state_dict()

    key_map = {
        "tok_embeddings.weight": "model.embed_tokens.weight",
        "norm.weight": "model.norm.weight",
        "output.weight": "lm_head.weight",
    }
    for i in range(args.n_layers):
        p = f"layers.{i}."
        key_map[p + "attention.wq.weight"] = f"model.layers.{i}.self_attn.q_proj.weight"
        key_map[p + "attention.wk.weight"] = f"model.layers.{i}.self_attn.k_proj.weight"
        key_map[p + "attention.wv.weight"] = f"model.layers.{i}.self_attn.v_proj.weight"
        key_map[p + "attention.wo.weight"] = f"model.layers.{i}.self_attn.o_proj.weight"
        key_map[p + "feed_forward.w1.weight"] = f"model.layers.{i}.mlp.gate_proj.weight"
        key_map[p + "feed_forward.w2.weight"] = f"model.layers.{i}.mlp.down_proj.weight"
        key_map[p + "feed_forward.w3.weight"] = f"model.layers.{i}.mlp.up_proj.weight"
        key_map[p + "attention_norm.weight"] = f"model.layers.{i}.input_layernorm.weight"
        key_map[p + "ffn_norm.weight"] = f"model.layers.{i}.post_attention_layernorm.weight"

    new_sd = {}
    for k, v in sd.items():
        nk = key_map.get(k)
        if nk is None:
            print("Mapping skipped:", k)
            continue
        if nk == "lm_head.weight" and TIE_WEIGHTS:
            continue
        new_sd[nk] = v.detach().to(torch.float16).contiguous().cpu()

    save_file(new_sd, str(out / "model.safetensors"), metadata={"format": "pt"})

    cfg = {
        "architectures": ["LlamaForCausalLM"],
        "model_type": "llama",
        "hidden_size": args.dim,
        "intermediate_size": int(2 * (4 * args.dim) / 3),
        "num_hidden_layers": args.n_layers,
        "num_attention_heads": args.n_heads,
        "num_key_value_heads": args.n_heads,
        "vocab_size": args.vocab_size,
        "max_position_embeddings": args.max_seq_len,
        "rope_theta": 10000.0,
        "rms_norm_eps": args.norm_eps,
        "hidden_act": "silu",
        "attention_bias": False,
        "mlp_bias": False,
        "tie_word_embeddings": bool(TIE_WEIGHTS),
        "torch_dtype": "float16",
        "bos_token_id": tokenizer.token_to_id("<bos>"),
        "eos_token_id": tokenizer.token_to_id("<eos>"),
        "pad_token_id": tokenizer.token_to_id("<pad>"),
    }
    (out / "config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    tokenizer.save(str(out / "tokenizer.json"))
    (out / "tokenizer_config.json").write_text(json.dumps({
        "bos_token": "<bos>", "eos_token": "<eos>", "pad_token": "<pad>",
        "unk_token": "<unk>", "model_max_length": args.max_seq_len,
        "tokenizer_class": "PreTrainedTokenizerFast",
    }, indent=2), encoding="utf-8")
    print(f"Exported to standard HF format: {out}")
    return out

MODEL_CARD = """---
license: apache-2.0
library_name: transformers
pipeline_tag: text-generation
model_name: Kotha-S1
company: Kotha Lab
creator: Md. Wahid
authors:
  - name: Md. Wahid
language:
  - bn
  - en
tags:
  - bengali
  - bangla
  - nlp
  - text-generation
  - llama-architecture
  - kotha
datasets:
  - wikimedia/wikipedia
  - ai4bharat/IndicCorpV2
  - HuggingFaceFW/fineweb-edu
  - local:bangladesh_1971_2026_dataset.json
---

# Kotha-S1

Bilingual Bengali and English model (~100M parameters) developed by Md. Wahid (Kotha Lab).
"""

try:
    export_dir = export_transformers_format(model, tokenizer, args)
    Path("README.md").write_text(MODEL_CARD, encoding="utf-8")
    for f in ["model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"]:
        upload_file(export_dir / f, f, f"Kotha-S1 weights — {f}")
    upload_file("README.md", "README.md", "Kotha-S1 model card")
    print(f"All artifacts uploaded successfully to Hugging Face repository: {HF_REPO_ID}")
except Exception as e:
    print(f"Error during export or upload: {e}")

In [ ]:
import torch
import torch.nn.functional as F

print("Chat Engine Initialized. Enter your prompt.")
print("(Type 'exit' or 'quit' to terminate)\n" + "-" * 50)

@torch.no_grad()
def chat_with_kotha(user_text, max_new_tokens=100, temperature=0.7, repetition_penalty=1.25):
    raw = model.module if hasattr(model, "module") else model
    raw.eval()

    prompt = f"[INST] {user_text.strip()} [/INST] "
    bos = tokenizer.token_to_id("<bos>")
    eos = tokenizer.token_to_id("<eos>")
    ids = ([bos] if bos is not None else []) + tokenizer.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device="cuda")

    generated = []
    for _ in range(max_new_tokens):
        x_cond = x[:, -args.max_seq_len:]
        with autocast_ctx():
            logits = raw(x_cond)[:, -1, :].float()
        for token_id in set(x[0].tolist()):
            if logits[0, token_id] < 0:
                logits[0, token_id] *= repetition_penalty
            else:
                logits[0, token_id] /= repetition_penalty

        logits = logits / max(temperature, 1e-5)
        probs = F.softmax(logits, dim=-1)

        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cdf = torch.cumsum(sorted_probs, dim=-1)
        sorted_probs = sorted_probs.masked_fill(cdf - sorted_probs > 0.9, 0.0)
        sorted_probs = sorted_probs / sorted_probs.sum()
        nxt = sorted_idx.gather(1, torch.multinomial(sorted_probs, num_samples=1))

        token = nxt.item()
        if token == eos:
            break
        generated.append(token)
        x = torch.cat((x, nxt), dim=1)

    reply = tokenizer.decode(generated).strip()
    if "[INST]" in reply:
        reply = reply.split("[INST]")[0].strip()
    return reply

while True:
    try:
        user_input = input("\nUser: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ["exit", "quit", "stop"]:
            print("\nChat ended.")
            break
        print("Kotha-S1 is thinking...")
        response = chat_with_kotha(user_input)
        print(f"Kotha-S1: {response}")
    except KeyboardInterrupt:
        print("\nChat interrupted.")
        break